# Import

In [ ]:
import numpy as np
import healpy as hp
import pandas as pd
import networkx as nx
import geopandas as gpd

from pint import UnitRegistry
from datetime import datetime
from tqdm.auto import tqdm
from itertools import combinations

from pyproj import Geod
from shapely import union_all, intersects, STRtree
from shapely.wkt import loads
from shapely.geometry import LineString
from healpix_geo.nested import healpix_to_lonlat
from healpix_geo.nested import lonlat_to_healpix

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt

ureg = UnitRegistry()

In [ ]:
def get_points(NSIDE):
    NPIX = hp.nside2npix(NSIDE)
    points = np.array(
        hp.pix2vec(NSIDE, range(NPIX), nest=True)
    ).T

    df = pd.DataFrame(points, columns=('x', 'y', 'z'))

    return df

def get_patches(NSIDE):
    NPIX = hp.nside2npix(NSIDE)

    pixels = np.arange(NPIX)
    patches = np.array(
        [hp.boundaries(NSIDE, i, nest=True).T for i in pixels]
    ).reshape(-1, 3)

    df = pd.DataFrame(patches, columns=('x', 'y', 'z'))

    return df

In [ ]:
def great_circle_path(a=None, b=None, n_points=100, iterable=False):
    geod = Geod(ellps='WGS84')

    if iterable:
        (a, b) = a

    points = geod.npts(
        a.x, a.y,
        b.x, b.y,
        n_points,
    )
    line = LineString(points)

    return line

def great_circle_distance(a=None, b=None, iterable=False):
    geod = Geod(ellps='WGS84')

    if iterable:
        (a, b) = a

    distance = geod.inv(
        a.x, a.y,
        b.x, b.y,
    )[2]

    return distance

# Data

## Borders

In [ ]:
kaz = gpd.read_file('data/geoBoundaries-KAZ-ADM0_simplified.geojson')
ukr = gpd.read_file('data/geoBoundaries-UKR-ADM1_simplified.geojson')
rus = gpd.read_file('data/geoBoundaries-RUS-ADM1_simplified.geojson')
blr = gpd.read_file('data/geoBoundaries-BLR-ADM1_simplified.geojson')
borders = pd.concat((kaz, ukr, rus, blr)).reset_index(drop=True)
borders = borders.to_crs(epsg=4326)
borders

## Airports

In [ ]:
airports = pd.read_csv('data/airports.csv')
airports = gpd.GeoDataFrame(
    airports.drop(['GeoPointLat', 'GeoPointLong'], axis=1),
    geometry=gpd.points_from_xy(
        *airports[['GeoPointLong', 'GeoPointLat']].values.T
    ), crs='EPSG:4326'
)
airports[airports.ICAO.isin(['KJFK', 'UAAA'])]

# HEALPix

In [ ]:
NSIDE = 16
healpix = get_points(NSIDE)

NPIX = hp.nside2npix(NSIDE)
ipix = np.arange(NPIX)
depth = np.log2(NSIDE).astype(int)

print(NPIX, depth)

In [ ]:
lon, lat = healpix_to_lonlat(ipix, depth=depth, ellipsoid='WGS84')
lon = (lon + 180) % 360 - 180

In [ ]:
borders_restricted = borders.loc[
    borders.shapeGroup.isin(['RUS', 'UKR', 'BLR']),
].reset_index(drop=True)

shape_restricted = union_all(
    borders_restricted.geometry
)

tree = STRtree(borders_restricted.geometry)

In [ ]:
start = lonlat_to_healpix(
    airports[airports.ICAO=='UAAA'].geometry.x,
    airports[airports.ICAO=='UAAA'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]
end = lonlat_to_healpix(
    airports[airports.ICAO=='KJFK'].geometry.x,
    airports[airports.ICAO=='KJFK'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]

In [ ]:
gdf = gpd.GeoDataFrame(
    healpix,
    geometry=gpd.points_from_xy(
        lon, lat
    ),
    crs='EPSG:4326',
)
gdf = gdf[~gdf.geometry.intersects(shape_restricted)]
gdf

# Graph

In [ ]:
import cudf
import cugraph

In [ ]:
G = nx.read_graphml('points_graph.graphml', node_type=int)

In [ ]:
avoid = True
counter = 0
start_time = datetime.now()

avg_pathfind_time = []
avg_intersect_time = []

#G.__networkx_cache__.clear()

while avoid:
    counter += 1

    pathfind_start = datetime.now()

    route = nx.dijkstra_path(
        G, start, end,
        backend='cugraph',
    )

    avg_pathfind_time.append(datetime.now() - pathfind_start)

    intersect = False

    intersect_start = datetime.now()

    for idx in range(len(route) - 1):
        point_a = route[idx]
        point_b = route[idx+1]

        coord_a = gdf.loc[point_a, 'geometry']
        coord_b = gdf.loc[point_b, 'geometry']

        #great_circle_start = datetime.now()

        path = great_circle_path(
            coord_a,
            coord_b,
            n_points=50,
        )

        #avg_great_circle_time.append(datetime.now() - great_circle_start)

        candidates_idx = tree.query(
            geometry=great_circle_path(
                *gdf.loc[[start, end], 'geometry'].values,
                n_points=50,
            ),
            predicate='intersects',
        )
        candidates = borders_restricted.loc[candidates_idx]

        for idx, candidate in candidates.iterrows():
            intersect = path.intersects(candidate.geometry)

            if intersect:
                avg_intersect_time.append(datetime.now() - intersect_start)
                G.remove_edge(point_a, point_b)
                break

        if intersect:
            break

    if counter % 25 == 0:
        elapsed = datetime.now() - start_time
        print(f'iterations: {counter}')
        print(f'time elapsed: {elapsed}')
        print(f'avg iteration time: {elapsed / counter}')
        print(f'avg pathfind time: {pd.to_timedelta(avg_pathfind_time).mean().to_pytimedelta()}')
        #print(f'avg great circle time: {pd.to_timedelta(avg_great_circle_time).mean().to_pytimedelta()}')
        print(f'avg intersect time: {pd.to_timedelta(avg_intersect_time).mean().to_pytimedelta()}\n')

    if not intersect:
        avoid = False

end_time = datetime.now()
elapsed = end_time - start_time

print(f'total iterations: {counter}\ntime elapsed: {elapsed}\navg iteration time: {elapsed / counter}')

In [ ]:
gdf.loc[route]

In [ ]:
gdf_route = gdf.loc[route].geometry.reset_index()
gdf_route = gdf_route.rename({'index': 'point_id'}, axis=1)
gdf_route.point_id = gdf_route.point_id.astype(str)

gdf_route.loc[gdf_route.point_id==str(start), 'point_id'] = 'UAAA'
gdf_route.loc[gdf_route.point_id=='UAAA', 'geometry'] = airports.loc[airports.ICAO=='UAAA', 'geometry'].values

gdf_route.loc[gdf_route.point_id==str(end), 'point_id'] = 'KJFK'
gdf_route.loc[gdf_route.point_id=='KJFK', 'geometry'] = airports.loc[airports.ICAO=='KJFK', 'geometry'].values

gdf_route

In [ ]:
countries = ['KAZ', 'UKR', 'RUS', 'BLR', 'USA']

fig = go.Figure()

trace = go.Choropleth(
    locations=countries,
    z=list(range(len(countries))),
    locationmode='ISO-3',
    colorscale=np.array(px.colors.qualitative.Plotly)[[2, 1, 1, 1, 4]],
    text=countries,
    showscale=False,
)
fig.add_trace(trace)

route_trace = go.Scattergeo(
    lon=gdf_route.geometry.x,
    lat=gdf_route.geometry.y,
    mode='lines',
    marker=dict(size=1, color='blue'),
)
fig.add_trace(route_trace)

start_end = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='lines+markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(start_end)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showland=True,
    ),
)
fig.show()

In [ ]:
nx.config.backend_priority = ['cugraph']

In [ ]:
G.__networkx_cache__.clear()

paths = nx.single_source_dijkstra(
    G, start,
)

In [ ]:
points_withtin_range = [
    key
    for key, value
    in paths[0].items()
    if (value * ureg.meter).to(ureg.nautical_mile).magnitude < 7e3
]

In [ ]:
countries = ['KAZ', 'UKR', 'RUS', 'BLR', 'USA']

fig = go.Figure()

trace = go.Choropleth(
    locations=countries,
    z=list(range(len(countries))),
    locationmode='ISO-3',
    colorscale=np.array(px.colors.qualitative.Plotly)[[2, 1, 1, 1, 4]],
    text=countries,
    showscale=False,
)
fig.add_trace(trace)

route_trace = go.Scattergeo(
    lon=gdf_route.geometry.x,
    lat=gdf_route.geometry.y,
    mode='lines',
    marker=dict(size=1, color='blue'),
)
fig.add_trace(route_trace)

start_end = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='lines+markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(start_end)

points = go.Scattergeo(
    lon=gdf.loc[points_withtin_range].geometry.x,
    lat=gdf.loc[points_withtin_range].geometry.y,
    mode='markers',
    marker=dict(size=2, color='black'),
)
fig.add_trace(points)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showland=True,
    ),
)
fig.show()

In [ ]:
edge_points_mask = np.isin(
    hp.get_all_neighbours(NSIDE, points_withtin_range, nest=True).T,
    points_withtin_range
).sum(axis=1)<7

edge_points = np.array(points_withtin_range)[edge_points_mask]
gdf.loc[edge_points]

In [ ]:
edge_points_neighbours = hp.get_all_neighbours(NSIDE, edge_points, nest=True).T

In [ ]:
neighbours_drop = edge_points[
    np.unique_counts(
        edge_points_neighbours[
            np.isin(edge_points_neighbours, edge_points)
        ]
    ).counts>2
]

In [ ]:
np.

In [ ]:
neighbours_drop

In [ ]:
{
    point: neighbour_ids[
        np.isin(neighbour_ids, edge_points)
        & ~np.isin(neighbour_ids, neighbours_drop)
    ]
    for point, neighbour_ids
    in zip(edge_points, edge_points_neighbours)
}

In [ ]:
np.unique_counts(edge_points_neighbours)

In [ ]:
{
    point: neighbour_ids[
        np.isin(neighbour_ids, edge_points)
    ]
    for point, neighbour_ids
    in zip(edge_points, edge_points_neighbours)
}

In [ ]:
countries = ['KAZ', 'UKR', 'RUS', 'BLR', 'USA']

fig = go.Figure()

trace = go.Choropleth(
    locations=countries,
    z=list(range(len(countries))),
    locationmode='ISO-3',
    colorscale=np.array(px.colors.qualitative.Plotly)[[2, 1, 1, 1, 4]],
    text=countries,
    showscale=False,
)
fig.add_trace(trace)

route_trace = go.Scattergeo(
    lon=gdf_route.geometry.x,
    lat=gdf_route.geometry.y,
    mode='lines',
    marker=dict(size=1, color='blue'),
)
fig.add_trace(route_trace)

start_end = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='lines+markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(start_end)

points = go.Scattergeo(
    lon=gdf.loc[edge_points].geometry.x,
    lat=gdf.loc[edge_points].geometry.y,
    mode='markers',
    marker=dict(size=2, color='black'),
)
fig.add_trace(points)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showland=True,
    ),
)
fig.show()

In [ ]:
route_segments = pd.DataFrame([
    {
        'segment': f'{window.point_id.values[0]} - {window.point_id.values[1]}',
        'distance_m': great_circle_distance(*window.geometry.values),
        'distance_nml': (
            great_circle_distance(*window.geometry.values) * ureg.meter
        ).to(ureg.nautical_mile).magnitude,
    }
    for window in gdf_route.rolling(2)
    if len(window) == 2
])
route_segments

In [ ]:
print(
    f'direct UAAA-KJFK: {(
        great_circle_distance(
            *gdf_route[
                gdf_route.point_id.isin(['UAAA', 'KJFK'])
            ].geometry.values
        ) * ureg.meter
    ).to(ureg.nautical_mile).magnitude:.2f} nautical miles'
)
print(f'segmented UAAA-KJFK: {route_segments.distance_nml.sum():.2f} nautical miles')